In [1]:
from google.colab import files
uploaded = files.upload()
!unzip combined.zip -d combined

Saving combined.zip to combined (16).zip
Archive:  combined.zip
replace combined/__MACOSX/._combined? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: combined/__MACOSX/._combined  
replace combined/combined/train.jsonl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: combined/combined/train.jsonl  
replace combined/__MACOSX/combined/._train.jsonl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: combined/__MACOSX/combined/._train.jsonl  
replace combined/combined/test.jsonl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: combined/combined/test.jsonl  
replace combined/__MACOSX/combined/._test.jsonl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: combined/__MACOSX/combined/._test.jsonl  
replace combined/combined/val.jsonl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: combined/combined/val.jsonl  
replace combined/__MACOSX/combined/._val.jsonl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: combined/__MACOSX/combined/._val.jsonl  


In [2]:
import pickle
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import LlamaTokenizerFast, LlamaForCausalLM, AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
from tqdm import tqdm
from google.colab import files
import random


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
!pip install bitsandbytes -q

In [4]:
# -----------------------------
# Dataset Class
# -----------------------------
class TranslationDataset(Dataset):
    """
    Handles tokenization and script tagging for combined native + romanized dataset.
    Each example has a 'script_id': 0=native, 1=romanized
    """
    def __init__(self, path, tokenizer, max_len=256):
        self.examples = []
        with open(path, "r") as f:
            for line in f:
                obj = json.loads(line)
                hi = obj["hi"].strip()
                en = obj["en"].strip()
                if not en:
                    continue

                # Determine script automatically: 0=Devanagari, 1=Romanized
                script_id = 0 if any('\u0900' <= c <= '\u097F' for c in hi) else 1

                tokenized = tokenizer(
                    hi,
                    text_target=en,
                    truncation=True,
                    max_length=max_len,
                    padding="max_length"
                )

                # Replace padding token in labels with -100 for loss
                tokenized["labels"] = [
                    l if l != tokenizer.pad_token_id else -100
                    for l in tokenized["labels"]
                ]

                tokenized["script_id"] = script_id
                tokenized["reference"] = en
                self.examples.append(tokenized)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
            "script_ids": torch.tensor(item["script_id"], dtype=torch.long),
            "reference": item["reference"]
        }

In [5]:
# -----------------------------
# Load Tokenizer and Base Model
# -----------------------------

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

print(f"Model: TinyLlama 1.1B (FP32 - STABLE)")
print(f"Model dtype: {model.dtype}")

Model: TinyLlama 1.1B (FP32 - STABLE)
Model dtype: torch.float32


In [6]:
# ------------------------------------------
# Load Datasets and DataLoaders
# ------------------------------------------

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = TranslationDataset("/content/combined/combined/train.jsonl", tokenizer, max_len=128)
test_dataset = TranslationDataset("/content/combined/combined/test.jsonl", tokenizer, max_len=128)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)


In [7]:
# -----------------------------
# PHASE 1: Train baseline WITH 8-bit Adam
# -----------------------------

print("=== PHASE 1: Training baseline (8-bit Adam) ===")

!pip install bitsandbytes -q  # Install if not already
from bitsandbytes.optim import Adam8bit

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

# Store original embeddings for later
original_embeddings = model.model.embed_tokens.weight.clone()

# 8-bit optimizer from the START
baseline_optimizer = Adam8bit(model.parameters(), lr=5e-6)  # 8-bit, not regular AdamW
num_stabilize_batches = 100

for batch_idx, batch in enumerate(tqdm(train_loader, desc="Stabilizing", total=num_stabilize_batches)):
    if batch_idx >= num_stabilize_batches:
        break

    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    baseline_optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

    loss = outputs.loss
    if torch.isnan(loss):
        print(f"⚠️ NaN in baseline training! Reducing LR...")
        for param_group in baseline_optimizer.param_groups:
            param_group['lr'] *= 0.5
        continue

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    baseline_optimizer.step()

    if batch_idx % 100 == 0:
        print(f"Baseline batch {batch_idx}: Loss = {loss.item():.4f}")
        torch.cuda.empty_cache()

print("✓ Baseline training complete with 8-bit Adam")
print(f"Memory after Phase 1: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

=== PHASE 1: Training baseline (8-bit Adam) ===


Stabilizing:   1%|          | 1/100 [00:00<01:09,  1.42it/s]

Baseline batch 0: Loss = 14.8521


Stabilizing: 100%|██████████| 100/100 [00:38<00:00,  2.60it/s]

✓ Baseline training complete with 8-bit Adam
Memory after Phase 1: 10.71GB


In [8]:
# -----------------------------
# PHASE 2: SETUP (KEEP USING 8-bit Adam) - FIXED VERSION
# -----------------------------

print("\n=== Phase 2: Continuing with 8-bit Adam ===")

class GradualScriptAdder(nn.Module):
    def __init__(self, trained_weights, hidden_size, vocab_size, num_scripts=2):
        super().__init__()
        # Create a NEW embedding layer with the trained weights
        self.token_embeddings = nn.Embedding(vocab_size, hidden_size)

        # Copy the trained weights into it
        with torch.no_grad():
            self.token_embeddings.weight.copy_(trained_weights)

        # Script bias
        self.script_bias = nn.Embedding(num_scripts, hidden_size)
        nn.init.zeros_(self.script_bias.weight)
        self.script_strength = 0.0
        self.update_step = 0

    def forward(self, input_ids, script_ids):
        token_emb = self.token_embeddings(input_ids)  # Now this works!
        if self.script_strength > 0:
            script_bias = self.script_bias(script_ids).unsqueeze(1)
            return token_emb + (script_bias * self.script_strength)
        return token_emb

    def increase_strength(self):
        self.update_step += 1
        self.script_strength = min(1.0, self.update_step / 2000)

# Get vocab size from model config
vocab_size = model.config.vocab_size
hidden_size = model.config.hidden_size

# Create and replace embeddings
gradual_adder = GradualScriptAdder(
    trained_weights=original_embeddings,  # The tensor of weights
    hidden_size=hidden_size,
    vocab_size=vocab_size,
    num_scripts=2
).to(device)

model.model.embed_tokens = gradual_adder

# CONTINUE with 8-bit Adam (same as Phase 1)
optimizer = Adam8bit(model.parameters(), lr=1e-5)

print("✓ Using 8-bit Adam for both phases")
print(f"✓ Memory before Phase 2: {torch.cuda.memory_allocated()/1024**3:.2f}GB")
print(f"✓ Script strength starts at: {gradual_adder.script_strength}")


=== Phase 2: Continuing with 8-bit Adam ===
✓ Using 8-bit Adam for both phases
✓ Memory before Phase 2: 10.95GB
✓ Script strength starts at: 0.0


In [9]:
# -----------------------------
# PHASE 2: Add script embeddings GRADUALLY - OPTIMIZED
# -----------------------------

print("\n=== PHASE 2: Adding script embeddings gradually (Optimized) ===")

import gc

# Use SIMPLE FP32 - faster and more stable than mixed precision
# REMOVED: scaler = torch.amp.GradScaler("cuda")
EPOCHS = 1

# Clear cache at start
torch.cuda.empty_cache()
gc.collect()

for epoch in range(EPOCHS):
    epoch_loss = 0
    num_batches = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        # 1. Move batch to device
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        script_ids = batch["script_ids"].to(device)

        # 2. Zero gradients
        optimizer.zero_grad()

        # 3. SIMPLE FP32 forward (no autocast overhead)
        inputs_embeds = model.model.embed_tokens(input_ids, script_ids)
        outputs = model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        # 4. SIMPLE FP32 backward (no scaler overhead)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        # REMOVED: scaler.scale(), .unscale_(), .step(), .update()

        # 5. Check for NaN and update script strength
        if torch.isnan(loss):
            print(f"⚠️ NaN detected! Reducing script strength...")
            gradual_adder.script_strength *= 0.5
            optimizer.zero_grad()
            continue

        # 6. Update script strength - ACCELERATED
        gradual_adder.increase_strength()
        # Boost script strength faster (safe acceleration)
        if gradual_adder.script_strength < 0.8:
            gradual_adder.script_strength += 0.0008

        # 7. Save loss value
        loss_value = loss.item()

        # 8. OPTIMIZED cleanup: less frequent to save time
        del inputs_embeds, outputs, loss
        if num_batches % 15 == 0:  # Clean every 15 batches instead of every batch
            gc.collect()
            torch.cuda.empty_cache()

        # 9. Track loss
        epoch_loss += loss_value
        num_batches += 1

        # 10. Progress update (less frequent to save I/O time)
        if num_batches % 1000 == 0:  # Was 500
            allocated = torch.cuda.memory_allocated() / 1024**3
            cached = torch.cuda.memory_reserved() / 1024**3
            print(f"Batch {num_batches}: Loss={loss_value:.4f}, "
                  f"Script strength={gradual_adder.script_strength:.3f}, "
                  f"GPU Mem: {allocated:.2f}GB allocated, {cached:.2f}GB reserved")

    avg_loss = epoch_loss / num_batches
    print(f"✓ Epoch {epoch+1}: Avg Loss={avg_loss:.4f}, "
          f"Script strength={gradual_adder.script_strength:.3f}")

    # Save checkpoint
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'script_adder_state': gradual_adder.state_dict(),
    }, "final_checkpoint.pt")
    print(f"✓ Checkpoint saved")

print("✓ Training complete!")


=== PHASE 2: Adding script embeddings gradually (Optimized) ===


Epoch 1:  10%|█         | 1000/9991 [06:44<57:13,  2.62it/s]

Batch 1000: Loss=9.0570, Script strength=0.501, GPU Mem: 13.36GB allocated, 13.67GB reserved


Epoch 1:  20%|██        | 2000/9991 [13:29<54:07,  2.46it/s]

Batch 2000: Loss=9.2002, Script strength=1.000, GPU Mem: 13.36GB allocated, 13.95GB reserved


...

Batch 8000: Loss=6.8316, Script strength=1.000, GPU Mem: 13.36GB allocated, 14.10GB reserved


Epoch 1:  90%|█████████ | 9000/9991 [1:00:40<06:20,  2.60it/s]

Batch 9000: Loss=7.0980, Script strength=1.000, GPU Mem: 13.36GB allocated, 14.10GB reserved


Epoch 1: 100%|██████████| 9991/9991 [1:07:21<00:00,  2.47it/s]


✓ Epoch 1: Avg Loss=7.2215, Script strength=1.000
✓ Checkpoint saved
✓ Training complete!


In [37]:
import shutil
from google.colab import files

torch.save(model.state_dict(), "trained_model.pt")
files.download("trained_model.pt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import sacrebleu
import json
from google.colab import files

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------
# 1️⃣ Upload your trained .pt file
# ---------------------------
uploaded = files.upload()  # select your trained_model.pt
pt_file = list(uploaded.keys())[0]

# ---------------------------
# 2️⃣ Load the wrapper weights
# ---------------------------
wrapper_state = torch.load(pt_file, map_location=device)

# ---------------------------
# 3️⃣ Extract original token embeddings
# ---------------------------
# The path inside your state dict looks like:
# "model.embed_tokens.original_layer...token_embeddings.weight"
# We find it dynamically:
embedding_key = [k for k in wrapper_state.keys() if "token_embeddings.weight" in k][0]
print(f"✓ Found token embeddings at: {embedding_key}")

# Extract the tensor
token_embeddings = wrapper_state[embedding_key].clone()

# ---------------------------
# 4️⃣ Load clean model
# ---------------------------
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device)

# Replace original embeddings with the extracted ones
with torch.no_grad():
    model.model.embed_tokens.weight.copy_(token_embeddings)

model.eval()
print("✓ Model loaded with extracted embeddings")

# ---------------------------
# 5️⃣ Upload your test JSONL
# ---------------------------
uploaded = files.upload()  # select your test.jsonl
test_file = list(uploaded.keys())[0]

# ---------------------------
# 6️⃣ Minimal test dataset
# ---------------------------
class TestDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=128):
        self.examples = []
        with open(path, "r") as f:
            for line in f:
                obj = json.loads(line)
                hi = obj["hi"].strip()
                en = obj["en"].strip()
                tokenized = tokenizer(
                    hi,
                    text_target=en,
                    truncation=True,
                    max_length=max_len,
                    padding="max_length"
                )
                tokenized["reference"] = en
                self.examples.append(tokenized)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "reference": item["reference"]
        }

test_dataset = TestDataset(test_file, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# ---------------------------
# 7️⃣ Generate predictions
# ---------------------------
predictions = []
references = []

for batch in tqdm(test_loader, desc="Generating translations"):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    references.append(batch["reference"][0])

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=48,
            num_beams=1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    pred_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predictions.append(pred_text)

# ---------------------------
# 8️⃣ Compute BLEU
# ---------------------------
bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"\n✓ BLEU score on {len(test_dataset)} examples: {bleu.score:.2f}")

# ---------------------------
# 9️⃣ Show a few examples
# ---------------------------
print("\nSome predictions vs references:")
for i in range(min(5, len(test_dataset))):
    input_text = tokenizer.decode(test_dataset[i]['input_ids'], skip_special_tokens=True)
    print(f"\nInput: {input_text}")
    print(f"Prediction: {predictions[i]}")
    print(f"Reference: {references[i]}")
